# 🎬 AI 4K Upscaler — Real-ESRGAN + RIFE
### Model 1: Fastest + Sharpest Quality Pipeline

**Pipeline:**
```
Input Video → RIFE v4.6 (fps boost) → Real-ESRGAN x4+ (4K upscale) → Output
```

| Step | What it does |
|------|--------------|
| ✅ RIFE v4.6 | Multiplies your FPS (e.g. 30 → 120 → 240) |
| ✅ Real-ESRGAN x4+ | Upscales to 4K with ultra-sharp AI detail |
| ✅ FP16 Half Precision | 2x faster on Colab T4 GPU |
| ✅ Tile Processing | Handles large frames without running out of VRAM |

> ⚡ **Make sure GPU is enabled:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — CHECK GPU
# ═══════════════════════════════════════════════════
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu}')
    print(f'✅ VRAM: {vram:.1f} GB')
else:
    print('❌ No GPU found! Go to Runtime → Change runtime type → GPU')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════
print('📦 Installing Real-ESRGAN...')
!pip install -q realesrgan basicsr facexlib gfpgan

print('📦 Installing RIFE (frame interpolation)...')
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python-headless ffmpeg-python tqdm Pillow

print('📦 Cloning RIFE v4.6...')
!git clone -q https://github.com/hzwer/Practical-RIFE.git /content/RIFE

print('✅ All dependencies installed!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — DOWNLOAD AI MODEL WEIGHTS
# ═══════════════════════════════════════════════════
import os

os.makedirs('/content/weights', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

# Download Real-ESRGAN x4+ weights (sharpest model)
print('⬇️  Downloading Real-ESRGAN x4+ weights...')
!wget -q -O /content/weights/RealESRGAN_x4plus.pth \
    https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth

# Download RIFE v4.6 weights
print('⬇️  Downloading RIFE v4.6 weights...')
os.makedirs('/content/RIFE/train_log', exist_ok=True)
!wget -q -O /content/rife_weights.pkl \
    https://github.com/hzwer/Practical-RIFE/releases/download/v4.6/flownet.pkl
!cp /content/rife_weights.pkl /content/RIFE/train_log/flownet.pkl

print('✅ All model weights ready!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — LOAD AI MODELS
# ═══════════════════════════════════════════════════
import sys
import torch
import numpy as np
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

sys.path.insert(0, '/content/RIFE')
from model.RIFE_HDv3 import Model as RIFEModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔧 Using device: {device}')

# ── Load Real-ESRGAN ──────────────────────────────
print('🧠 Loading Real-ESRGAN x4+...')
esrgan_model = RRDBNet(
    num_in_ch=3, num_out_ch=3,
    num_feat=64, num_block=23,
    num_grow_ch=32, scale=4
)
upsampler = RealESRGANer(
    scale=4,
    model_path='/content/weights/RealESRGAN_x4plus.pth',
    model=esrgan_model,
    tile=512,          # tile size — reduce to 256 if VRAM issues
    tile_pad=10,
    pre_pad=0,
    half=True,         # FP16 = 2x faster
    device=device
)
print('✅ Real-ESRGAN loaded!')

# ── Load RIFE ─────────────────────────────────────
print('🧠 Loading RIFE v4.6...')
rife_model = RIFEModel()
rife_model.load_model('/content/RIFE/train_log', -1)
rife_model.eval()
rife_model.device()
print('✅ RIFE loaded!')

print('\n🚀 Both AI models ready!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 — HELPER FUNCTIONS
# ═══════════════════════════════════════════════════
import cv2
from PIL import Image
from tqdm import tqdm


def rife_interpolate(frame1, frame2, model, scale=1.0, times=1):
    """Interpolate frames between frame1 and frame2 using RIFE."""
    def to_tensor(f):
        f = torch.from_numpy(f).permute(2, 0, 1).float() / 255.0
        return f.unsqueeze(0).to(device)

    I0 = to_tensor(frame1)
    I1 = to_tensor(frame2)
    middle = model.inference(I0, I1, scale)
    mid = (middle[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    return mid


def upscale_frame_esrgan(frame, upsampler):
    """Upscale a single frame with Real-ESRGAN."""
    output, _ = upsampler.enhance(frame, outscale=4)
    return output


def get_video_info(path):
    """Get fps, width, height from a video file."""
    cap = cv2.VideoCapture(path)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, width, height, frames


print('✅ Helper functions ready!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 — UPLOAD YOUR VIDEO
# ═══════════════════════════════════════════════════
from google.colab import files

print('📁 Upload your video file (MP4, MKV, AVI, MOV)...')
uploaded = files.upload()

INPUT_VIDEO = list(uploaded.keys())[0]
INPUT_PATH  = f'/content/{INPUT_VIDEO}'

fps, w, h, total_frames = get_video_info(INPUT_PATH)

print(f'\n📹 Video Info:')
print(f'   Resolution : {w}x{h}')
print(f'   FPS        : {fps}')
print(f'   Frames     : {total_frames}')
print(f'\n📤 Output will be: {w*4}x{h*4} @ up to {int(fps)*4}fps')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 — ⚙️ SETTINGS (Edit these!)
# ═══════════════════════════════════════════════════

# ── FPS Multiplier ────────────────────────────────
# 2 = doubles fps  (30 → 60)
# 4 = quadruples   (30 → 120)
# 8 = 8x           (30 → 240)
FPS_MULTIPLIER = 4

# ── Upscale Scale ─────────────────────────────────
# 4 = 4K output (recommended)
# 2 = 2K output (faster)
UPSCALE_FACTOR = 4

# ── Processing Mode ───────────────────────────────
# 'both'   = RIFE + ESRGAN (best quality, slower)
# 'esrgan' = Only upscale (no fps boost)
# 'rife'   = Only fps boost (no upscale)
MODE = 'both'

# ── Tile size (reduce if VRAM errors) ─────────────
TILE_SIZE = 512

OUTPUT_PATH = '/content/output/upscaled_4k.mp4'

target_fps = int(fps * FPS_MULTIPLIER)
print(f'⚙️  Settings confirmed:')
print(f'   Mode          : {MODE}')
print(f'   FPS           : {int(fps)} → {target_fps}')
print(f'   Resolution    : {w}x{h} → {w*UPSCALE_FACTOR}x{h*UPSCALE_FACTOR}')
print(f'   Output path   : {OUTPUT_PATH}')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8 — 🚀 RUN THE FULL PIPELINE
# ═══════════════════════════════════════════════════
import subprocess
import tempfile

cap = cv2.VideoCapture(INPUT_PATH)

# Temp folder for processed frames
frames_dir = '/content/frames'
os.makedirs(frames_dir, exist_ok=True)

frame_idx   = 0
prev_frame  = None

print(f'🎬 Processing {total_frames} frames...')
pbar = tqdm(total=total_frames, unit='frame')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ── Step 1: RIFE interpolation ──────────────────
    if MODE in ('both', 'rife') and prev_frame is not None:
        for t in range(1, FPS_MULTIPLIER):
            interp = rife_interpolate(prev_frame, frame_rgb, rife_model)

            if MODE == 'both':
                interp_bgr = cv2.cvtColor(interp, cv2.COLOR_RGB2BGR)
                interp_up  = upscale_frame_esrgan(interp_bgr, upsampler)
                save_frame = interp_up
            else:
                save_frame = cv2.cvtColor(interp, cv2.COLOR_RGB2BGR)

            cv2.imwrite(f'{frames_dir}/{frame_idx:08d}.png', save_frame)
            frame_idx += 1

    # ── Step 2: Real-ESRGAN upscale ─────────────────
    if MODE in ('both', 'esrgan'):
        frame_up = upscale_frame_esrgan(frame, upsampler)
        cv2.imwrite(f'{frames_dir}/{frame_idx:08d}.png', frame_up)
    else:
        cv2.imwrite(f'{frames_dir}/{frame_idx:08d}.png', frame)

    frame_idx  += 1
    prev_frame  = frame_rgb
    pbar.update(1)

cap.release()
pbar.close()
print(f'\n✅ Frames processed: {frame_idx} total')

# ── Step 3: Reassemble with FFmpeg ──────────────────
print('🎞️  Assembling final video with FFmpeg...')
!ffmpeg -y -framerate {target_fps} \
    -i {frames_dir}/%08d.png \
    -c:v libx264 -crf 14 -preset slow \
    -pix_fmt yuv420p \
    {OUTPUT_PATH} -loglevel error

print(f'\n🎉 DONE! Output saved to: {OUTPUT_PATH}')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 9 — PREVIEW OUTPUT
# ═══════════════════════════════════════════════════
from IPython.display import HTML
from base64 import b64encode

# Show a short preview (first 5 seconds)
preview_path = '/content/output/preview.mp4'
!ffmpeg -y -i {OUTPUT_PATH} -t 5 -c copy {preview_path} -loglevel error

mp4 = open(preview_path, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()

HTML(f'''
<h3>🎬 Preview (first 5 seconds)</h3>
<video controls width="100%" style="border-radius:12px;max-width:900px">
  <source src="{data_url}" type="video/mp4">
</video>
''')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 10 — DOWNLOAD YOUR 4K VIDEO
# ═══════════════════════════════════════════════════
from google.colab import files
import os

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'📦 Output file size: {size_mb:.1f} MB')
print('⬇️  Starting download...')

files.download(OUTPUT_PATH)

print('✅ Download complete!')

---
## 🛠️ Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Lower `TILE_SIZE` to 256 in Cell 7 |
| Slow processing | Set `MODE = 'esrgan'` to skip RIFE |
| Blurry output | Make sure `half=True` and tile_pad=10 |
| RIFE error | Re-run Cell 3 to re-download weights |
| No GPU | Runtime → Change runtime type → T4 GPU |

## ⚡ Speed Tips
- Use **T4 GPU** (free) or **A100** (Colab Pro) for best speed
- Shorter clips process faster — trim before uploading
- Set `FPS_MULTIPLIER = 2` for faster processing
- Set `UPSCALE_FACTOR = 2` for 2K instead of 4K (4x faster)

---
*Built with Real-ESRGAN x4+ + RIFE v4.6 — Model 1 Pipeline*